In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from aml_detector.rules.repeat_sender_rule import RepeatSenderRule

data = pd.read_parquet("../data/processed/transactions_clean.parquet")
print(data.shape)

(5078345, 13)


In [2]:
rule = RepeatSenderRule(max_transactions=20)
flagged = rule.evaluate(data)
print(f"Accounts flagged: {len(flagged)}")
flagged.head(10)

Evaluating Accounts: 100%|██████████| 74353/74353 [46:37<00:00, 26.58it/s]   

Accounts flagged: 74353


,account,total_transactions,laundering_transactions,rule_id,severity
0,100428660,168672,243,VELOCITY_001,high
1,1004286A8,103018,158,VELOCITY_001,high
2,100428978,20497,29,VELOCITY_001,high
3,80266F880,98,29,VELOCITY_001,high
4,100428810,16426,26,VELOCITY_001,high
5,812D22980,27,25,VELOCITY_001,high
6,100428738,13756,23,VELOCITY_001,high
7,811C597B0,37,21,VELOCITY_001,high
8,1004286F0,18663,21,VELOCITY_001,high
9,100428780,17264,21,VELOCITY_001,high


In [3]:
total_laundering = data["Is Laundering"].sum()
caught_laundering = flagged["laundering_transactions"].sum()

print(f"Total laundering transactions in data: {total_laundering}")
print(f"Laundering transactions inside flagged accounts: {caught_laundering}")
print(f"Recall: {caught_laundering / total_laundering:.2%}")

Total laundering transactions in data: 5177
Laundering transactions inside flagged accounts: 2362
Recall: 45.62%


In [4]:
# Flagged accounts that had ZERO laundering
clean_flagged = flagged[flagged["laundering_transactions"] == 0]
print(f"Flagged accounts with no laundering: {len(clean_flagged)}")
print(f"Flagged accounts with laundering:    {len(flagged) - len(clean_flagged)}")
print(f"Precision (accounts): {(len(flagged) - len(clean_flagged)) / len(flagged):.2%}")

Flagged accounts with no laundering: 73484
Flagged accounts with laundering:    869
Precision (accounts): 1.17%


In [ ]:
import pandas as pd

# Flagged accounts ranked by laundering count
print("Distribution of laundering transactions per flagged account:")
print(flagged["laundering_transactions"].value_counts().sort_index(ascending=False).head(20))

Distribution of laundering transactions per flagged account:
laundering_transactions
243     1
158     1
29      2
26      1
25      1
23      1
21      5
18      2
17      3
16      9
15      9
14      7
13     16
12      7
11      3
10      2
9       3
8       1
7       2
6       3
Name: count, dtype: int64
